# 01. JAX 基本 — jax.numpy / 配列の不変性 / PRNG キー

**このノートブックの内容**:
- `jax.numpy` (jnp) は NumPy の置換 (ほぼ同 API)
- JAX 配列は **不変** (immutable)
- 乱数は **明示的な PRNG キー** で管理
- `jit` の予告編 (詳細は 03_jit_vmap.ipynb)

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (01_basics.md)](../01_basics.md)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)

print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
%matplotlib inline

## 1. `jax.numpy` は NumPy 互換

ほとんどの NumPy API は jnp でも同じ名前で使える。

In [ ]:
# NumPy 版
x_np = np.array([1.0, 2.0, 3.0])
print(f'NumPy:   sum = {np.sum(x_np)}, mean = {np.mean(x_np)}')

# JAX 版 (jnp)
x_jax = jnp.array([1.0, 2.0, 3.0])
print(f'JAX:     sum = {jnp.sum(x_jax)}, mean = {jnp.mean(x_jax)}')

print(f'\n型: NumPy {type(x_np).__name__}, JAX {type(x_jax).__name__}')
print(f'JAX 配列は GPU/TPU 対応の DeviceArray')

## 2. 不変性 (Immutability) — JAX の最重要ルール

NumPy では `x[0] = 99` で書き換えできるが、JAX 配列は **書き換え禁止**。
代わりに **`.at[i].set(value)`** で「新しい配列を返す」 操作を行う。

In [ ]:
# NumPy: 直接書き換え OK
x_np = np.array([1.0, 2.0, 3.0])
x_np[0] = 99
print(f'NumPy 書き換え後: {x_np}')

# JAX: 直接書き換えは TypeError
x_jax = jnp.array([1.0, 2.0, 3.0])
try:
    x_jax[0] = 99
except TypeError as e:
    print(f'\nJAX 書き換え TypeError: {str(e)[:80]}...')

# 正しい書き方: .at[].set()
x_new = x_jax.at[0].set(99.0)
print(f'\nJAX 正しい更新:')
print(f'  元の配列: {x_jax}')
print(f'  新しい配列: {x_new}')
print('  ↑ 元の x_jax は変わらない (関数型プログラミング的)')

## 3. 乱数 — 明示的 PRNG キー

NumPy: `np.random.normal(...)` グローバル状態を使う
JAX:   `jax.random.normal(key, ...)` キーを毎回渡す

→ JAX の方式は **再現性** と **並列化** に強い。

In [ ]:
# JAX 乱数の基本パターン
key = jax.random.PRNGKey(42)

# 同じキーなら何度生成しても同じ値
sample1 = jax.random.normal(key, shape=(3,))
sample2 = jax.random.normal(key, shape=(3,))
print(f'同じキー → 同じ値:')
print(f'  {sample1}')
print(f'  {sample2}')

# 違う値が欲しいときは split で派生キーを作る
key, subkey = jax.random.split(key)
sample3 = jax.random.normal(subkey, shape=(3,))
print(f'\nsplit 後の派生キーで生成:\n  {sample3}')

## 4. 配列の dtype デフォルトは float32

NumPy は float64 (倍精度) デフォルト、JAX は float32 (単精度) デフォルト。
ML では float32 で十分なケースが多く、メモリ・速度で有利。

In [ ]:
x_np = np.array([1.0, 2.0])
x_jax = jnp.array([1.0, 2.0])
print(f'NumPy dtype: {x_np.dtype}')   # float64
print(f'JAX dtype:   {x_jax.dtype}')  # float32

# 倍精度が必要なら明示的に
from jax import config
# config.update('jax_enable_x64', True)  # 全体で float64 に切替 (起動時のみ有効)
x_jax_64 = jnp.array([1.0, 2.0], dtype=jnp.float64)
print(f'JAX float64 明示: {x_jax_64.dtype}')

## 5. `@jit` の予告編 — 高速化

詳細は [`03_jit_vmap.ipynb`](03_jit_vmap.ipynb) で扱いますが、雰囲気だけ。

In [ ]:
import time

def slow_fn(x):
    for _ in range(100):
        x = jnp.sin(x) + jnp.cos(x)
    return x

# 通常実行
fast_fn = jax.jit(slow_fn)  # JIT コンパイル版

x = jnp.linspace(0, 1, 10_000)

# 普通実行
t0 = time.time()
_ = slow_fn(x).block_until_ready()
print(f'通常:  {time.time() - t0:.3f} 秒')

# JIT (初回はコンパイル時間込み、2 回目以降が速い)
_ = fast_fn(x).block_until_ready()  # warm up
t0 = time.time()
_ = fast_fn(x).block_until_ready()
print(f'JIT:   {time.time() - t0:.3f} 秒')

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [章 TOP](../README.md) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [02_autodiff.ipynb](02_autodiff.ipynb) |